# BB scattering analysis: Part 1

On this notebook, you will start your analysis of your results of the BB scattering experiment. Your *entire* analysis should be done with Python. You don't want to be moving between a spreadsheet and your scripts. In order to do that, it is best practice to start from the raw data and do all of your calculations in your script. 

For the BB scattering experiment, you were counting the number of impacts per 5-cm bin:

![bins](bins.png)

The important numbers to write down are:
- The size of the bins. This is a single number as you can assume they are all the same size.
- The number of impacts in each bin. This will be an array.
- The radius of the detector.
- The horizontal distance of the gun when the screw is turned 10 times.

Start your script in the cell below. Make sure to:
- Import numpy
- Import matplotlib
- Add the `"%matplotlib inline"` command so that the figures appear in the notebook.
- Assign the size of your bins to `dx`.
- Assign the number of impacts in each bin to the array `num_events`.
- Assign the radius of the detector to `r_detector`.
- Assign the horizontal distance to `h`.

In [ ]:
# Your code goes here

## Goals of the analysis

Analyzing data is the process of answering specific questions about a system. That's not to say that you can't explore a dataset for new or unexpected patterns. But even in those cases, you need to know what you are trying to calculate.

*The purpose of this experiment is to determine the size of the hidden target ($R$) under the plastic shield in the middle of the apparatus.*

But there are other things we can look at to develop some intuition about the system.

## Plotting the raw data

Before we try to find $R$, we can plot the raw data to see if anything interesting emerges. In our case, a good place to start would be to make a bar graph of the number of events in each bin. Why a bar graph?
- Each bin is the same width.
- Each bin contains the same information (number of hits).
- A bar graph will convey the spread of the data across the circumference.

You could likely convey the same information with a scatter plot.

In the cell below, create a bar graph of `num_events`. Making the figure will have a similar syntax as what you have done before but using the `bar` function (documentation [here](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html)) instead of `plot`. Notice that the `bar` function requires both a x- and y-axis. Here, create a simple array of numbers from 1 to $N$ where $N$ is the number of bins in your data. That array will be your x-axis.

At this point, you don't need to label the axis or make an effort to make the graph look pretty. You just want a sense of your data.

In [ ]:
# Your code goes here

### Checkpoint

You should see the number of impacts increases for higher numbered bins. In the lab, this means that higher angles saw more hits. But your data is also likely very "choppy". This is due to the relatively low number of events in the whole experiment. The theory for this experiment definitely predicts a smooth relationship between the number of hits and the angle which should be reflected here. Call me over to look at your results.

### Your observations

In the cell below confirm the probable observations made above. Comment on what else you are noticing.

**Answer:** ...

### Improving the statistics

Ask for the results from 1 other team. Assign those values to `num_events_2`. Everything else should be the same. Create an variable:

`num_events_tot = num_events + num_events_2`

And then make a new graph of your data.

In [ ]:
# Your code goes here.

Comments on those results and how they compare to your previous figure.

**Answer:** ...

## Comparing your results to the model

If you haven't already, take the time to review the model for scattering from circular targets described in the lab handout. The final equation in the derivation is given below:
$$
N = \Phi \frac{R}{2}\sin\left(\frac{\theta}{2}\right)\Delta\theta
$$
It might look complicated but you only need geometry and calculus to follow the derivation. In the this equation:
- $N$ is the number of scattering events.
- $\Phi$ is the flux.
- $R$ is the radius of the target.
- $\theta$ is the angle of the scattering events.
- $\Delta\theta$ is the interval in $\theta$ over which the events are added.

Our data allows us to know $N$, $\Phi$, $\theta$, and $\Delta\theta$. The only unknown in that equation is $R$, the radius of the scattering target that we are looking for. There are different approaches we can take here. We will work through the first one together and it will be up to you to develop the second approach.

### $N$ vs $\theta$

We'll first generate a plot of $N$ as a function of $\theta$ and use that to determine the value of $R$. We need to derive some of those variables:
- From the size of the bins ($dx$), we can calculate $\Delta\theta$.

$$
\Delta\theta = dx/r\_detector
$$

- From the horizontal distance ($h$) of the gun for 10 turns, we can calculate the linear flux $\Phi$. Since you fired 10 BB's every turn (5 every half turn), the flux during your experiment was $\Phi = 100/h$. If you decide to use the combined datasets (your plus another team), then your effective flux was twice as high: $\Phi = 200/h$. 
- From the bin number and the bin size, we can calculate the angle of each bin:

$$
\theta = (bin*dx - 0.025)/r\_detector
$$

The equations above work if all the distance values are in the same units. We recommend you use meters. The code below implements these definitions.

In [ ]:
Dtheta = dx/r_detector
flux = 200/h
theta = (bin_num*dx-0.025)/r_detector

print(Dtheta * 180/np.pi)
print(theta.size)
print(num_events_tot.size)

We added a couple of print statements to check our results. First, we get a $\Delta\theta = 8.7$ degrees. We have 17 bins. It means that the total angle over which we measured our results is $17*8.7 \approx 150$ degrees. This seems reasonable. (Or at least, it does not raise any alarm flag.)

Next, we verified that there is the same number of points in the `theta` and the `num_events_tot` arrays. 

We can generate a plot of those data.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
ax.scatter(theta, num_events_tot)

#### Fit

To get the value of $R$, we can fit a curve to those data. There are multiple libraries to fit functions inside of Python. Many of them are in a library called SciPy (Scientific Python). Today, we will use the function `curve_fit` (documentation [here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)) to find R.

First, we have to define the function that we wish to fit to:

In [ ]:
def scattering(theta, R):
    N = flux*R/2*np.sin(theta/2)*Dtheta
    return N

In the function above, the values of `flux` and `Dtheta` are set prior in the code. The first argument of the function definition is the independent variable (`theta`) and all following arguments (there can be more than one) are fit parameters. Here, our only fit parameter is `R`. 

With that function defined, we can run the fit:

In [ ]:
from scipy.optimize import curve_fit
popt, pcov = curve_fit(scattering, theta, num_events_tot)

The arguments to `curve_fit` are the function to fit, the independent variable (x-axis), and the dependent variable (y-axis). The algorithm returns the value of `R` so that the function matches the data the best. Those parameters are hidden in `popt` and `pcov`. `popt` is an array whose elements correspond to the fit variables. Here `popt` has a single element and `R` is equal to `popt[0]`. The uncertainty in the fit parameter can be extracted from `pcov`. The code below demonstrates this:

In [ ]:
R = popt[0]
dR = np.sqrt(np.diag(pcov))[0]
print(R)
print(dR)

We can visualize how good the fit is by plotting the `scattering` function on top of our data:

In [ ]:
theta_fit = np.linspace(theta[0], theta[-1], 1000)
num_events_fit = scattering(theta_fit, R)

fig = plt.figure()
ax = fig.add_subplot(111)
ax.scatter(theta, num_events_tot, c='b', label="Data")
ax.plot(theta_fit, num_events_fit, c='r', label="Fit")
ax.set_xlabel("Angle (rad)")
ax.set_ylabel("Number of scattering event")
ax.legend(loc='upper left')

We could play more tricks here. We could plot fit lines at `R+dR` and `R-dR` to show the area within $\pm$one standard deviation. We could try different fitting algorithm. And so on. 

For now, take a few minutes to comment on yoru value of `R` and `dR`.

**Answer:** ...

### $N$ vs $\sin(\theta/2)$

A different way to reproduce this analysis is to plot `num_events_tot` as a function of $\sin(\theta/2)$ instead of just $\theta$. Go ahead and perform this analysis here. You will have to change the definition of the `scattering` (or create a new fit function) since you are changing the independent variable.

In [ ]:
# Your code goes here

Comments on your results and how they compare to the previous method.

**Answer:** ...

## Best visual representation

As a last exercise, we should look at all the figures we generated today and decide which one is best (if any) to communicate our result. Go through the checklist below and answer each question.

1. What is the purpose of the analysis? What are you looking for?

**Answer:** ...

2. Of the 3 figures you created today, which one(s) help you communicate that result?

**Answer:** ...

3. If you selected more than one figure, is there one you think communicates the result better? Why?

**Answer:** ...

4. Are there things you feel would improve the visual presentation of the data for the figure you selected? Even if you do not know how to accomplish those with matplotlib yet?

**Answer:** ...

If you have time left in this lab, it would be good for you to try to implement some of the improvements that you listed above. You can look online for help on coding some of the features. You can also ask the instructor for hints.

In [ ]:
# The code for your final figure goes here.